In [ ]:
# Celda 1 - clonar spacemoe o traer los archivos nuevos
# Si no esta: clona. Si ya esta: busca actualizaciones y trae solo lo que
# cambio (fast-forward), y lista los archivos nuevos.
import os
import subprocess

REPO = "https://github.com/eldeeon20/spacemoe.git"
RAMA = "main"
DESTINO = "spacemoe"


def sh(*args, cwd=None):
    print("$", " ".join(args))
    r = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    if r.stdout.strip():
        print(r.stdout.strip())
    if r.returncode != 0:
        print("ERROR:", r.stderr.strip())
    return r


if not os.path.isdir(os.path.join(DESTINO, ".git")):
    print("no estaba: clonando")
    sh("git", "clone", "--branch", RAMA, "--depth", "1", REPO, DESTINO)
else:
    sh("git", "fetch", "--depth", "1", "origin", RAMA, cwd=DESTINO)
    antes = sh("git", "rev-parse", "HEAD", cwd=DESTINO).stdout.strip()
    sh("git", "merge", "--ff-only", f"origin/{RAMA}", cwd=DESTINO)
    despues = sh("git", "rev-parse", "HEAD", cwd=DESTINO).stdout.strip()
    if antes == despues:
        print("SIN CAMBIOS: ya estaba al dia")
    else:
        print("ACTUALIZADO: archivos nuevos o modificados")
        print("A  nuevo | M  modificado | D  borrado")
        sh("git", "diff", "--name-status", antes, despues, cwd=DESTINO)

print("\narchivos en", DESTINO)
for nombre in sorted(os.listdir(DESTINO))[:40]:
    print(" ", nombre)

In [ ]:
# Celda 2 - entrenar
# Corre train.py de spacemoe. Poner flags en ARGS si hace falta.
import subprocess
from pathlib import Path

DESTINO = globals().get("DESTINO", "spacemoe")
DIR = Path(DESTINO)
ARGS = []  # por ejemplo: ["--steps", "1000"]

if not (DIR / "train.py").exists():
    raise SystemExit(f"no esta {DIR}/train.py: corré la celda 1 primero")

print("entrenando en", DIR.resolve())
r = subprocess.run(["python", "train.py", *ARGS], cwd=DIR)
print("codigo de salida:", r.returncode)